This is the production version of your Bronze notebook. It will:

- Read widgets — environment and run_mode
- Load config from pipeline_config.yml
- Ingest all 8 CSVs to Bronze Delta tables
- Run data quality checks
- Log the run to the audit table

In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])
dbutils.widgets.dropdown("run_mode", "full", ["full", "incremental"])

env = dbutils.widgets.get("environment")
print(env)
run_mode = dbutils.widgets.get("run_mode")
print(run_mode)

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/data_quality.py

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/helpers.py

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
       StructType,StructField,
       StringType,
       IntegerType,
       LongType,
       TimestampType,
       DoubleType
       )

configs = get_config(env)
print(configs)
base_path = configs["base_path"]
layers = configs["layers"]


In [0]:
## Defining Paths
RAW_PATH    = f"{base_path}/raw"
BRONZE_PATH = f"{base_path}/bronze"
SILVER_PATH = f"{base_path}/silver"
GOLD_PATH   = f"{base_path}/gold"
AUDIT_PATH  = f"{base_path}/audit"

print(f"RAW    → {RAW_PATH}")
print(f"BRONZE → {BRONZE_PATH}")
print(f"SILVER → {SILVER_PATH}")
print(f"GOLD   → {GOLD_PATH}")
print(f"AUDIT  → {AUDIT_PATH}")

In [0]:

orders_schema = StructType([
    StructField('order_id', StringType()),
    StructField('customer_id', StringType()),
    StructField('order_status', StringType()),
    StructField('order_purchase_timestamp', TimestampType()),
    StructField('order_approved_at', TimestampType()),
    StructField('order_delivered_carrier_date', TimestampType()),
    StructField('order_delivered_customer_date', TimestampType()),
    StructField('order_estimated_delivery_date', TimestampType()),
])

customers_schema = StructType([
    StructField('customer_id', StringType()),
    StructField('customer_unique_id', StringType()),
    StructField('customer_zip_code_prefix', IntegerType()),
    StructField('customer_city', StringType()),
    StructField('customer_state', StringType()),
])

products_schema = StructType([
    StructField('product_id', StringType()),
    StructField('product_category_name', StringType()),
    StructField('product_name_lenght', IntegerType()),
    StructField('product_description_lenght', IntegerType()),
    StructField('product_photos_qty', IntegerType()),
    StructField('product_weight_g', IntegerType()),
    StructField('product_length_cm', IntegerType()),
    StructField('product_height_cm', IntegerType()),
    StructField('product_width_cm', IntegerType()),
])

sellers_schema = StructType([
    StructField('seller_id', StringType()),
    StructField('seller_zip_code_prefix', IntegerType()),
    StructField('seller_city', StringType()),
    StructField('seller_state', StringType()),
])

payments_schema = StructType([
    StructField('order_id', StringType()),
    StructField('payment_sequential', IntegerType()),
    StructField('payment_type', StringType()),
    StructField('payment_installments', IntegerType()),
    StructField('payment_value', DoubleType()),
])

order_items_schema = StructType([
    StructField('order_id', StringType()),
    StructField('order_item_id', IntegerType()),
    StructField('product_id', StringType()),
    StructField('seller_id', StringType()),
    StructField('shipping_limit_date', TimestampType()),
    StructField('price', DoubleType()),
    StructField('freight_value', DoubleType()),
])

reviews_schema = StructType([
    StructField('review_id', StringType()),
    StructField('order_id', StringType()),
    StructField('review_score', IntegerType()),
    StructField('review_comment_title', StringType()),
    StructField('review_comment_message', StringType()),
    StructField('review_creation_date', TimestampType()),
    StructField('review_answer_timestamp', TimestampType()),
])

translation_schema = StructType([
    StructField('product_category_name', StringType()),
    StructField('product_category_name_english', StringType()),
])

schemas = {
    "orders": orders_schema,
    "customers": customers_schema,
    "products": products_schema,
    "sellers": sellers_schema,
    "payments": payments_schema,
    "order_items": order_items_schema,
    "reviews": reviews_schema,
    "translation": translation_schema
}


In [0]:
from datetime import datetime
start_time = datetime.now()
total_rows = 0

for table in layers["bronze"]:
    name = table["name"]
    source = table["source"]
    primary_key = table["primary_key"]
    schema = schemas[name]
    
    # 1. read CSV — use spark.read.csv with RAW_PATH and source
    df = spark.read.csv(f"{RAW_PATH}/{source}",schema=schema,header=True)
    # 2. add ingestion_date and source_file_name columns
    df = df.withColumn("ingestion_date", F.current_timestamp()) \
                     .withColumn("source_file_name",F.lit(source))
    # 3. write to bronze using write_delta()
    write_delta(df, f"{BRONZE_PATH}/{name}", "overwrite", partition_col=None)
    total_rows += df.count()
    # 4. run data quality checks using run_all_checks()
    if primary_key:
        run_all_checks(df, name, primary_key, [primary_key])
    else:
        check_row_count(df, name)
    
    print(f"{name} ingested successfully")


In [0]:
end_time = datetime.now()

log_run(
    spark=spark,
    notebook_name="job_01_ingest",
    environment=env,
    run_mode=run_mode,
    start_time=start_time,
    end_time=end_time,
    rows_processed=total_rows,
    status="SUCCESS",
    audit_path=AUDIT_PATH,
    error_message=None
)


In [0]:
spark.read.format("delta").load(f"{AUDIT_PATH}/pipeline_runs").display()

In [0]:
dbutils.notebook.exit("SUCCESS")